In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip "/content/drive/My Drive/Visdrone Dataset/VisDrone2019-DET-train.zip" -d /content/visdrone_DET

Streaming output truncated to the last 5000 lines.
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000140.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000141.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000142.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000143.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000144.jpg  
  inflating: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000145.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000146.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000147.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000148.jpg  
 extracting: /content/visdrone_DET/VisDrone2019-DET-train/images/9999937_00000_d_0000149.jpg  

In [ ]:
!unzip "/content/drive/My Drive/Visdrone Dataset/VisDrone2019-DET-val.zip" -d /content/visdrone_DET

Archive:  /content/drive/My Drive/Visdrone Dataset/VisDrone2019-DET-val.zip
   creating: /content/visdrone_DET/VisDrone2019-DET-val/
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/.DS_Store  
   creating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/0000001_02999_d_0000005.txt  
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/0000001_03499_d_0000006.txt  
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/0000001_03999_d_0000007.txt  
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/0000001_04527_d_0000008.txt  
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/0000001_05249_d_0000009.txt  
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/0000001_05499_d_0000010.txt  
  inflating: /content/visdrone_DET/VisDrone2019-DET-val/annotations/0000001_05999_d_0000011.txt  
  inflating: /content/visdrone_DET/VisDrone2

In [ ]:
train_dir = '/content/visdrone_DET/VisDrone2019-DET-train'
val_dir = '/content/visdrone_DET/VisDrone2019-DET-val'


In [ ]:
import os
import cv2
from PIL import Image
import pandas as pd
def visdrone_to_yolo(img_dir,ann_dir,yolo_ann_dir):
    os.makedirs(yolo_ann_dir, exist_ok=True)
    # loop that goes through annotation directory
    for ann_file in os.listdir(ann_dir):
        if not ann_file.endswith('.txt'):
            continue

        img_file = ann_file.replace('.txt','.jpg')
        img_path = os.path.join(img_dir, img_file)
        ann_path = os.path.join(ann_dir, ann_file)
        yolo_ann_path = os.path.join(yolo_ann_dir, ann_file)

        if not os.path.exists(img_path):
            print(f"Image does not exist for annotation: {ann_file}, skipping.")
            continue


        img = Image.open(img_path)
        w, h = img.size

        yolo_ann_lines = []
        with open(ann_path,'r')as file:
            for lines in file:
                parts = lines.strip().split(',')
                if len(parts) < 8:
                    continue
                x, y, width, height, score, category, trunc, occ = map(float, parts[:8])
                category = int(category)

                # skip ignored regions
                if category == 0:
                    continue

                yolo_cls = category - 1
                x_center =  (x + width/2 ) / w
                y_center =  (y + height/2 ) / h
                norm_width = width / w
                norm_height = height / h

                yolo_ann_lines.append(f"{yolo_cls} {x_center:.6f} {y_center:.6f} {norm_width:.6f} {norm_height:.6f}")

        if yolo_ann_lines:
            with open(yolo_ann_path, 'w') as file:
                file.write('\n'.join(yolo_ann_lines))
        else:
            print(f"Skipping empty label file: {ann_file}")



visdrone_to_yolo(
    ann_dir=os.path.join(train_dir, 'annotations'),
    img_dir=os.path.join(train_dir, 'images'),
    yolo_ann_dir= os.path.join(train_dir,'labels')
)

visdrone_to_yolo(
    ann_dir=os.path.join(val_dir, 'annotations'),
    img_dir=os.path.join(val_dir, 'images'),
    yolo_ann_dir= os.path.join(val_dir,'labels')
)


In [ ]:
import os

os.makedirs('/content/dataset', exist_ok=True)

yaml_content = """
path: /content/visdrone_DET
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images

names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""

with open('/content/dataset/visdrone.yaml', 'w') as f:
    f.write(yaml_content)


In [ ]:
pip install -U ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:
pip install git+https://github.com/ultralytics/ultralytics.git@main

  Cloning https://github.com/ultralytics/ultralytics.git (to revision main) to /tmp/pip-req-build-pxfym5xz
  Running command git clone --filter=blob:none --quiet https://github.com/ultralytics/ultralytics.git /tmp/pip-req-build-pxfym5xz
  Resolved https://github.com/ultralytics/ultralytics.git to commit b905114f46311976c816d586073579a1a0b29266
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os

def clean_label_dir(labels_dir, max_class_idx=9):     #Removes lines from label files where the class index exceeds max_class_idx.


     for fname in os.listdir(labels_dir):
        if not fname.endswith('.txt'):
            continue
        fpath = os.path.join(labels_dir, fname)
        with open(fpath, 'r') as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            try:
                class_idx = int(line.split()[0])
                if class_idx <= max_class_idx:
                    new_lines.append(line)
            except Exception as e:
                print(f"Skipping line in {fname}: {line.strip()} (Error: {e})")
        with open(fpath, 'w') as f:
            f.writelines(new_lines)


train_labels_dir = '/content/visdrone_DET/VisDrone2019-DET-train/labels'
val_labels_dir = '/content/visdrone_DET/VisDrone2019-DET-val/labels'
clean_label_dir(train_labels_dir, max_class_idx=9)
clean_label_dir(val_labels_dir, max_class_idx=9)

In [ ]:
from ultralytics import YOLOWorld

project_dir = '/content/drive/MyDrive/yolov8s_worldv2'

# Load model
model = YOLOWorld("yolov8s-worldv2.pt")
'''
for param in model.model.text_encoder.parameters():
    param.requires_grad = False

frozen = all(not param.requires_grad for param in model.model.text_encoder.parameters())
print("Text encoder frozen:", frozen)
'''
# Set VisDrone classes
model.set_classes([
    'pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor'
])

# Fine-tuning
results = model.train(
    data='/content/dataset/visdrone.yaml',
    freeze='text_encoder',
    epochs=100,
    imgsz=928,
    batch=8,
    lr0=0.0002,
    patience=10,            # Early stopping patience
    augment=True,           # Enable all augmentations
    mosaic=1.0,             # Strong mosaic augmentation
    save=True,
    save_period=5,
    project=project_dir,
    name='runs',
    exist_ok=True,
    workers=8

)


KeyboardInterrupt: 

In [ ]:
from ultralytics import YOLOWorld

model = YOLOWorld("/content/drive/MyDrive/yolov8s_worldv2/runs/weights/last.pt")
results = model.train(resume=True)

Ultralytics 8.3.169 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/visdrone.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=text_encoder, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=928, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/yolov8s_worldv2/runs/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=runs, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patien

train: Scanning /content/visdrone_DET/VisDrone2019-DET-train/labels.cache... 6471 images, 0 backgrounds, 0 corrupt: 100%|██████████| 6471/6471 [00:00<?, ?it/s]

train: /content/visdrone_DET/VisDrone2019-DET-train/images/0000137_02220_d_0000163.jpg: 1 duplicate labels removed
train: /content/visdrone_DET/VisDrone2019-DET-train/images/0000140_00118_d_0000002.jpg: 1 duplicate labels removed
train: /content/visdrone_DET/VisDrone2019-DET-train/images/9999945_00000_d_0000114.jpg: 1 duplicate labels removed
train: /content/visdrone_DET/VisDrone2019-DET-train/images/9999987_00000_d_0000049.jpg: 1 duplicate labels removed
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
Reading existed cache from '/content/visdrone_DET/VisDrone2019-DET-train/text_embeddings_clip_ViT-B_32.pt'


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1223.3±929.6 MB/s, size: 131.6 KB)


val: Scanning /content/visdrone_DET/VisDrone2019-DET-val/labels.cache... 548 images, 0 backgrounds, 0 corrupt: 100%|██████████| 548/548 [00:00<?, ?it/s]


...

     36/100      11.3G      1.223     0.8453     0.9303        466        928: 100%|██████████| 809/809 [05:39<00:00,  2.38it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:08<00:00,  4.12it/s]


                   all        548      38759      0.568      0.458      0.477      0.294

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     37/100      12.3G      1.222     0.8414     0.9276        659        928: 100%|██████████| 809/809 [05:36<00:00,  2.40it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:09<00:00,  3.80it/s]


                   all        548      38759      0.573      0.452      0.476      0.295

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     38/100      7.69G      1.219     0.8386     0.9297        542        928:  57%|█████▋    | 465/809 [03:13<02:21,  2.43it/s]

In [ ]:
from ultralytics import YOLOWorld

model = YOLOWorld("/content/drive/MyDrive/yolov8s_worldv2/runs/weights/last.pt")
results = model.train(resume=True)

Ultralytics 8.3.169 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/visdrone.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=text_encoder, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=928, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/yolov8s_worldv2/runs/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=runs, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patien

train: Scanning /content/visdrone_DET/VisDrone2019-DET-train/labels.cache... 6471 images, 0 backgrounds, 0 corrupt: 100%|██████████| 6471/6471 [00:00<?, ?it/s]

train: /content/visdrone_DET/VisDrone2019-DET-train/images/0000137_02220_d_0000163.jpg: 1 duplicate labels removed
train: /content/visdrone_DET/VisDrone2019-DET-train/images/0000140_00118_d_0000002.jpg: 1 duplicate labels removed
train: /content/visdrone_DET/VisDrone2019-DET-train/images/9999945_00000_d_0000114.jpg: 1 duplicate labels removed
train: /content/visdrone_DET/VisDrone2019-DET-train/images/9999987_00000_d_0000049.jpg: 1 duplicate labels removed
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
Reading existed cache from '/content/visdrone_DET/VisDrone2019-DET-train/text_embeddings_clip_ViT-B_32.pt'


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1223.3±929.6 MB/s, size: 131.6 KB)


val: Scanning /content/visdrone_DET/VisDrone2019-DET-val/labels.cache... 548 images, 0 backgrounds, 0 corrupt: 100%|██████████| 548/548 [00:00<?, ?it/s]


...

     36/100      11.3G      1.223     0.8453     0.9303        466        928: 100%|██████████| 809/809 [05:39<00:00,  2.38it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:08<00:00,  4.12it/s]


                   all        548      38759      0.568      0.458      0.477      0.294

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     37/100      12.3G      1.222     0.8414     0.9276        659        928: 100%|██████████| 809/809 [05:36<00:00,  2.40it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:09<00:00,  3.80it/s]


                   all        548      38759      0.573      0.452      0.476      0.295

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     38/100      7.69G      1.219     0.8386     0.9297        542        928:  57%|█████▋    | 465/809 [03:13<02:21,  2.43it/s]

In [ ]:
from ultralytics import YOLOWorld

model = YOLOWorld("/content/drive/MyDrive/yolov8s_worldv2/runs/weights/last.pt")
results = model.train(resume=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.169 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/visdrone.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=text_encoder, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=928, int8=False, iou=0.7, keras=False, k

100%|██████████| 755k/755k [00:00<00:00, 21.3MB/s]


Overriding model.yaml nc=80 with nc=10

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytic

100%|██████████| 5.35M/5.35M [00:00<00:00, 79.6MB/s]


AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 66.8±30.8 MB/s, size: 261.9 KB)


...

                   all        548      38759        0.6      0.467      0.499       0.31

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     82/100      11.1G      1.123     0.7049     0.9011        569        928: 100%|██████████| 809/809 [05:18<00:00,  2.54it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:08<00:00,  3.90it/s]


                   all        548      38759      0.607      0.459      0.499      0.311
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 72, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

45 epochs completed in 4.103 hours.
Optimizer stripped from /content/drive/MyDrive/yolov8s_worldv2/runs/weights/last.pt, 25.8MB
Optimizer stripped from /content/drive/MyDrive/yolov8s_worldv2/runs/weights/best.pt, 25.8MB

Validating /content/drive/MyDrive/yolov8s_worldv2/runs/weights/best.pt...
Ultralytics 8.3.169 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8s-worldv2 summary (fused): 87 layers, 12,749,288 parameters, 0 gradients, 34.2 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:14<00:00,  2.37it/s]


                   all        548      38759      0.595      0.471      0.499      0.311
            pedestrian        520       8844      0.643      0.498       0.56      0.279
                people        482       5125      0.637      0.381       0.44      0.189
               bicycle        364       1287      0.431      0.254      0.262      0.126
                   car        515      14064      0.786       0.83      0.856      0.629
                   van        421       1975      0.561      0.523      0.535      0.391
                 truck        266        750      0.578      0.439      0.461      0.327
              tricycle        337       1045      0.515      0.385      0.384      0.225
       awning-tricycle        220        532      0.397      0.248      0.241      0.151
                   bus        131        251      0.796       0.61      0.674      0.511
                 motor        485       4886      0.609       0.54      0.575       0.28
Speed: 0.4ms preproce